In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense
from tensorflow.keras.callbacks import EarlyStopping

# 1. Data Preparation (using your dataHandling function)
def dataHandling(df, sensorUsed=None, scaler=None, normalize=True, windowSize=30):
    sensorUsed = sensorUsed or ['s' + str(i) for i in range(1, 22)]
    segmentedData, engineIds, beginnings = [], [], []
    for engId in df['engineId'].unique():
        arr = df[df['engineId'] == engId][sensorUsed].values
        for start in range(0, len(arr) - windowSize + 1):
            segmentedData.append(arr[start:start+windowSize])
            engineIds.append(engId)
            beginnings.append(start+1)
    X = np.array(segmentedData)
    if normalize:
        flat = X.reshape(-1, X.shape[-1])
        scaler = scaler or MinMaxScaler()
        flat_scaled = scaler.fit_transform(flat)
        X = flat_scaled.reshape(X.shape)
    return X, engineIds, beginnings, scaler

dir = '~/Desktop/Dropbox/1.codeData/turbofanEngines/data/'
# Load data
colnames = ["engineId","cycle","setting1","setting2","setting3"] + ['s'+str(i) for i in range(1,22)]
df_train = pd.read_csv(dir + 'df_train.txt', sep='\s+', names=colnames, header=None)
df_test  = pd.read_csv(dir + 'df_test.txt',  sep='\s+', names=colnames, header=None)
df_true  = pd.read_csv(dir + 'df_true.txt',                 names=['RUL'], header=None)

# Compute RUL and cap
df_train['rul'] = df_train.groupby('engineId')['cycle'].transform(lambda x: x.max() - x)
df_train['rul'] = df_train['rul'].clip(upper=130)
df_train['rul'] = df_train['rul'] / 130  # normalize RUL for training

# Create sliding windows for features and RUL
windowSize = 30
X_train, train_ids, train_starts, scaler = dataHandling(df_train, 
                                                        sensorUsed=['s'+str(i) for i in range(1,22)],
                                                        windowSize=windowSize)
# For each window, RUL is the last timestep's RUL (normalized)
rul_vals = df_train.groupby('engineId')['rul'].apply(lambda g: g.values)
y_train = []
for eng, start in zip(train_ids, train_starts):
    y_train.append(rul_vals.loc[eng][start + windowSize - 2])  # offset to index
y_train = np.array(y_train)

# Test windows: only last window per engine
X_test_windows, test_ids, test_starts, _ = dataHandling(df_test, 
                                                         sensorUsed=['s'+str(i) for i in range(1,22)],
                                                         scaler=scaler, normalize=True,
                                                         windowSize=windowSize)
# pick last window for each engine
last_indices = {}
for i, eid in enumerate(test_ids):
    last_indices[eid] = i
print(last_indices)
X_test_last = np.array([X_test_windows[idx] for idx in last_indices.values()])
print(X_test_last.shape)

y_test = df_true['RUL'].values / 130  # normalized

# 2. Build Seq2Seq Autoencoder
seq_in = Input(shape=(windowSize, X_train.shape[2]))
# Encoder
encoded, state_h, state_c = LSTM(64, return_state=True)(seq_in)
encoder_states = [state_h, state_c]

# Decoder
dec_input = RepeatVector(windowSize)(encoded)
dec_lstm = LSTM(64, return_sequences=True)(dec_input)
decoded = TimeDistributed(Dense(X_train.shape[2]))(dec_lstm)

autoencoder = Model(seq_in, decoded)
autoencoder.compile(optimizer='adam', loss='mse')

# Train autoencoder
early = EarlyStopping(patience=5, restore_best_weights=True)
autoencoder.fit(X_train, X_train,
                epochs=100,
                batch_size=32,
                validation_split=0.1,
                callbacks=[early],
                verbose=2)

# 3. Build RUL predictor using encoder
encoder_model = Model(seq_in, encoded)

# Freeze encoder weights
encoder_model.trainable = False

# RUL regression head
reg_input = Input(shape=(windowSize, X_train.shape[2]))
enc_out = encoder_model(reg_input)
reg_out = Dense(1, activation='linear')(enc_out)
reg_model = Model(reg_input, reg_out)
reg_model.compile(optimizer='adam', loss='mse')

# Train regression head
reg_model.fit(X_train, y_train,
              epochs=50,
              batch_size=16,
              validation_split=0.1,
              callbacks=[early],
              verbose=2)

# 4. Predict & Evaluate
y_pred = reg_model.predict(X_test_last).flatten()
# Rescale back to original RUL scale
y_pred_rescaled = y_pred * 130
y_test_rescaled = y_test * 130
rmse = np.sqrt(mean_squared_error(y_test_rescaled, y_pred_rescaled))
print(f"Autoencoder‐decoder Test RMSE: {rmse:.2f}")


2025-06-20 09:16:39.401157: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-20 09:16:39.536662: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-20 09:16:40.110755: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/home/ducan/anaconda3/envs/tensorflow2.10.0Crusher/lib/
2025-06-20 09:16:40.110847: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerro

{1: 1, 2: 21, 3: 118, 4: 195, 5: 264, 6: 340, 7: 471, 8: 608, 9: 634, 10: 797, 11: 851, 12: 1039, 13: 1205, 14: 1222, 15: 1269, 16: 1353, 17: 1489, 18: 1593, 19: 1699, 20: 1854, 21: 1973, 22: 1983, 23: 2084, 24: 2241, 25: 2260, 26: 2307, 27: 2418, 28: 2547, 29: 2689, 30: 2803, 31: 2970, 32: 3086, 33: 3107, 34: 3281, 35: 3450, 36: 3547, 37: 3639, 38: 3735, 39: 3743, 40: 3847, 41: 3941, 42: 4068, 43: 4211, 44: 4236, 45: 4359, 46: 4476, 47: 4520, 48: 4569, 49: 4843, 50: 4888, 51: 5003, 52: 5163, 53: 5298, 54: 5390, 55: 5474, 56: 5581, 57: 5712, 58: 5859, 59: 5924, 60: 6042, 61: 6172, 62: 6375, 63: 6501, 64: 6640, 65: 6682, 66: 6800, 67: 6842, 68: 7000, 69: 7025, 70: 7148, 71: 7187, 72: 7289, 73: 7372, 74: 7480, 75: 7539, 76: 7715, 77: 7848, 78: 7891, 79: 7963, 80: 8067, 81: 8251, 82: 8384, 83: 8428, 84: 8571, 85: 8576, 86: 8657, 87: 8684, 88: 8723, 89: 8871, 90: 8988, 91: 9193, 92: 9314, 93: 9529, 94: 9633, 95: 9693, 96: 9761, 97: 9866, 98: 9958, 99: 10026, 100: 10195}
(100, 30, 21)


2025-06-20 09:16:41.436768: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-06-20 09:16:41.437008: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-06-20 09:16:41.443201: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-06-20 09:16:41.443398: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-06-20 09:16:41.443559: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from S

Epoch 1/100


2025-06-20 09:16:45.550479: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8100
2025-06-20 09:16:46.208513: I tensorflow/stream_executor/cuda/cuda_blas.cc:1614] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


KeyboardInterrupt: 